# Part 4 — Heterogeneous queries

Each query below joins several layers. Across separate signaling, regulatory, and
complex tables, each join requires manual bookkeeping. Against the single graph,
each query is a few lines.

In [1]:
import pandas as pd

import uc2_common as uc

G = uc.load()
E = uc.edges_frame(G)
V = G.views.vertices().to_pandas()

## Q1 — Transcription factors that co-regulate complex assembly

This query joins three layers: complex hyperedges (protein ∈ complex), translation
coupling (gene → protein), and regulatory edges (TF → target). It keeps each
transcription factor whose targets cover at least half of a complex's subunits. It
returns 1,290 (TF, complex) pairs from 2,691 complexes of three or more
subunits.

In [2]:
subunits = uc.complex_subunits(G, min_size=3)
reg = E[E.edge_kind == "regulatory"]
tf_targets = reg.assign(tf=reg.source.str.removeprefix("gene:"),
                        tgt=reg.target.str.removeprefix("gene:")).groupby("tf")["tgt"].apply(set)

pairs = [{"tf": tf, "complex_id": cid, "regulated": len(h := genes & tgts), "subunits": len(genes),
          "fraction": round(len(h) / len(genes), 3)}
         for cid, genes in subunits.items() for tf, tgts in tf_targets.items()
         if len(genes & tgts) / len(genes) >= 0.5 and len(genes & tgts) >= 2]
coreg = pd.DataFrame(pairs).sort_values(["fraction", "regulated"], ascending=False)
print(f"complexes (>=3 subunits): {len(subunits):,} | TFs: {len(tf_targets):,} | pairs (>=50%): {len(pairs):,}")
print(coreg.head(10).to_string(index=False))

complexes (>=3 subunits): 2,691 | TFs: 367 | pairs (>=50%): 1,290
   tf                                            complex_id  regulated  subunits  fraction
 MYCN                                               cpx:MCM          6         6       1.0
 TP53 cpx:MSH2-MLH1-PMS2-PCNA DNA-repair initiation complex          4         4       1.0
CEBPA                    cpx:Cell cycle kinase complex CDK2          4         4       1.0
  MYC                    cpx:Cell cycle kinase complex CDK4          4         4       1.0
STAT1               cpx:PLC-gamma-2-SLP-76-Lyn-Grb2 complex          4         4       1.0
 ETS1                     cpx:SMAD3-SMAD4-cJun-cFos complex          4         4       1.0
  SP1                     cpx:SMAD3-SMAD4-cJun-cFos complex          4         4       1.0
 MYCN                       cpx:MCM2-MCM4-MCM6-MCM7 complex          4         4       1.0
 ETS1                    cpx:CyclinD3-CDK4-CDK6-p21 complex          4         4       1.0
 E2F4                   

## Q2 — Cross-compartment signaling, and Q3 — hub complexes

Q2 keeps signaling edges whose two endpoints sit in different organelle
compartments. Q3 ranks complexes by the summed signaling degree of their member
proteins.

In [3]:
comp = dict(zip(V.vertex_id, V.compartment))
sig = E[E.edge_kind == "signaling"].assign(sc=lambda d: d.source.map(comp), tc=lambda d: d.target.map(comp))
cross = sig[sig.sc != sig.tc]
print(f"cross-compartment signaling: {len(cross):,} / {len(sig):,} ({len(cross)/max(len(sig),1):.1%})")
print(cross.groupby(["sc", "tc"]).size().sort_values(ascending=False).head(6).to_string())

deg = pd.concat([sig.source.value_counts(), sig.target.value_counts()], axis=1).fillna(0).sum(axis=1)
hubs = pd.DataFrame([
    {"complex_id": cid, "n_members": len(m), "agg_signaling_degree": float(sum(deg.get(f"prot:{g}", 0) for g in m))}
    for cid, m in uc.complex_subunits(G).items()]).sort_values("agg_signaling_degree", ascending=False)
hubs.head(10).to_csv(uc.TABLES / "hub_complexes.csv", index=False)
print("\ntop hub complexes:\n", hubs.head(5).to_string(index=False))

cross-compartment signaling: 19,283 / 31,898 (60.5%)
sc  tc
n   c     3988
c   n     3571
    p     1979
p   c     1404
n   p     1021
v   c      676

top hub complexes:
                                complex_id  n_members  agg_signaling_degree
cpx:VEGFR2-S1PR3-ERK1/2-PKC-alpha complex          5                1258.0
cpx:VEGFR2-S1PR1-ERK1/2-PKC-alpha complex          5                1256.0
cpx:VEGFR2-S1PR2-ERK1/2-PKC-alpha complex          5                1253.0
cpx:VEGFR2-S1PR5-ERK1/2-PKC-alpha complex          5                1247.0
              cpx:CDC2-CCNA2-CDK2 complex          3                1105.0
